In [0]:
from pyspark.sql import DataFrame
from pyspark.sql.functions import *

# -----------------------------
# 1. Get vars (widgets + defaults)
# -----------------------------

def get_widget_or_default(name: str, default: str) -> str:
    try:
        return dbutils.widgets.get(name)
    except Exception:
        return default

DEFAULT_CATALOG = "eligibility_operation"
DEFAULT_SCHEMA = "default"
DEFAULT_INPUT_BASE = "/Volumes/eligibility_operation/default/input_files"
DEFAULT_OUTPUT_BASE = "/Volumes/eligibility_operation/default/output_files"
DEFAULT_PARTNER = "bettercare"
DEFAULT_TARGET_TABLE = "member_eligibility_staging"

catalog = get_widget_or_default("catalog", DEFAULT_CATALOG)
schema = get_widget_or_default("schema", DEFAULT_SCHEMA)
input_file_base_path = get_widget_or_default("input_file_base_path", DEFAULT_INPUT_BASE)
output_file_base_path = get_widget_or_default("output_file_base_path", DEFAULT_OUTPUT_BASE)
partner_code = get_widget_or_default("partner", DEFAULT_PARTNER)
output_table_target = get_widget_or_default("output_table_target", DEFAULT_TARGET_TABLE)

db = f"{catalog}.{schema}"
partner_config_tbl = f"{db}.partner_config"
partner_colmap_tbl = f"{db}.partner_column_mapping"
target_table = f"{db}.{output_table_target}"

# -----------------------------
# 2. Get delimiter + partner_id
# -----------------------------

def get_partner_config(partner_code: str):
    cfg = (
        spark.table(partner_config_tbl)
        .filter(col("partner_code") == partner_code.upper())
        .select("partner_id", "file_delimiter")
        .limit(1)
    )
    rows = cfg.collect()
    if not rows:
        raise ValueError(f"No config for partner_code={partner_code} in {partner_config_tbl}")
    r = rows[0]
    return r["partner_id"], r["file_delimiter"]

partner_id, delimiter = get_partner_config(partner_code)

# -----------------------------
# 3. Read raw partner file(s)
# -----------------------------

def read_partner_files(partner_code: str, base_path: str, delimiter: str) -> DataFrame:
    path = f"{base_path}/{partner_code}/*"
    return (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .option("delimiter", delimiter)
        .csv(path)
    )

raw_df = read_partner_files(partner_code, input_file_base_path, delimiter)

# display(raw_df)
# -----------------------------
# 4. Build rename mapping from partner_column_mapping
# -----------------------------
# Assume partner_column_mapping has:
#   partner_id, source_column_name, target_column_name

colmap_df = (
    spark.table(partner_colmap_tbl)
    .filter(col("partner_id") == partner_id)
    .select("source_column_name", "target_field_name")
)

mapping = {
    r["source_column_name"]: r["target_field_name"]
    for r in colmap_df.collect()
}

# Apply renames only for columns that exist in the raw_df
renamed_df = raw_df
for src_col, tgt_col in mapping.items():
    if src_col in renamed_df.columns:
        renamed_df = renamed_df.withColumnRenamed(src_col, tgt_col)

# Optionally add partner_id, validation_status, etc. expected in staging
renamed_df = renamed_df \
    .withColumn("partner_id", lit(partner_id)) \
    .withColumn("validation_status", lit("PENDING ")) \
    .withColumn("validation_errors", lit("Unknown")) \
        .withColumn("partner_code", lit(partner_code))

display(renamed_df)

# -----------------------------
# 5. Write into member_eligibility_staging
# -----------------------------

(
    renamed_df.write.format("delta")
    .mode("append")
    .saveAsTable(target_table)
)

external_id,first_name,last_name,dob,email,phone,partner_id,validation_status,validation_errors,partner_code
BC-001,Alice,Johnson,1965-08-10,alice.j@test.com,5552223333,2,PENDING,Unknown,bettercare
BC-002,Charlie,Brown,1972-03-25,charlie.b@test.com,5554445555,2,PENDING,Unknown,bettercare
null,David,Wilson,1958-12-20,david.wilson@example.com,5556667777,2,PENDING,Unknown,bettercare
BC-004,Emma456,Garcia,1990/06/15,emma.garcia@mail.com,555-888-9999,2,PENDING,Unknown,bettercare
BC-005,FRANK,LOPEZ,1945-03-30,FRANK.LOPEZ@TEST.COM,+15551112222,2,PENDING,Unknown,bettercare
BC-006,"Maria, Jr.",Rodriguez,1982-07-25,maria.rodriguez@health.org,5553334444,2,PENDING,Unknown,bettercare
NULL,George,Martinez,1800-01-01,george@,123,2,PENDING,Unknown,bettercare
BC-008,Hannah,Anderson,1995-11-18,hannah.anderson@example.com,5557778888,2,PENDING,Unknown,bettercare
BC-001,Isaac,Thompson,2025-12-31,isaac.thompson@test.com,(555) 999-0000,2,PENDING,Unknown,bettercare
BC-010,Julia,White,1963-04-22,julia.white@mail.com,5552221111,2,PENDING,Unknown,bettercare
